# SHACL 1.2 Node Expressions — User Guide

SHACL 1.2 Node Expressions is an expression language for computing nodes inside a shapes graph. 

Node expressions are used in SHACL in several places:
 - `sh:targetNode` (which nodes a shape applies to), 
 - `sh:expression` (a boolean conformance check), 
 - `sh:values` (a computed property value), 
 - `sh:deactivated` (whether a shape is active at all), and 
 - `sh:rule`'s `sh:subject`/`sh:predicate`/`sh:object` (deriving new triples).

This guide builds up from simple node expression to the full vocabulary, one new concept at a time. 

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the main user guide).
2. Run cells from top to bottom — later sections reuse the running "Person" example data from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")

## 1. Selecting target nodes

A node expression can be used as the object of a shape's sh:targetNode, to select the list of targets for a a shape.  

In this example we also introduce `shnex:instancesOf` which operates within a node expression similar to sh:targetClass.  

**Example 1**

In [2]:
#all ex:Person must have at least one ex:email. 
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:PersonShape a sh:NodeShape ;
      sh:targetNode [ shnex:instancesOf ex:Person ] ;
      sh:property [ sh:path ex:email ; sh:minCount 1 ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms, "- expect False (alice and bob both targeted, neither has ex:email)")
print("violations:", result.report_text.count("Focus Node:"))

conforms: False - expect False (alice and bob both targeted, neither has ex:email)
violations: 2


### 1.1 Narrowing the target set

Using `shnex:instancesOf` alone does not provide additional flexibility beyone using is a plain `sh:targetClass`.   

Added flexibility shows up once you *combine* expressions. `shnex:filterShape` narrows a candidate set down to only the nodes that conform to a given shape, computing a target set that a fixed `sh:target*` predicate could not express directly. `shnex:nodes` is used in conjunction with `shnex:filterShape` to select the nodes to be filtered.

**Example 1.1**

In [3]:
#all ex:Person 18 and over must have at least one ex:email. 
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 25 .
    ex:bob a ex:Person ; ex:age 12 .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:AdultShape a sh:NodeShape ;
      sh:targetNode [ shnex:nodes [ shnex:instancesOf ex:Person ] ;
                      shnex:filterShape [ sh:property [ sh:path ex:age ; sh:minInclusive 18 ] ]
                    ] ;
      sh:property [ sh:path ex:email ; sh:minCount 1 ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms, "- expect False (only adult alice is targeted; bob, a minor, is never checked)")
print("violations:", result.report_text.count("Focus Node:"))

conforms: False - expect False (only adult alice is targeted; bob, a minor, is never checked)
violations: 1


## 2. Constraint checks: using `sh:expression`

`sh:expression` requires a node expression to evaluate to exactly `(true)` for the shape to conform — a way to write a constraint using an arbitrary boolean condition with node expressions. 

For this example, the object of `sh:expression` is set to `true`.  The literal `true` is a literal node expression.  In later examples we will use other node expression functions to compute the value of the `sh:expression`.

Note that sh:expression true and sh:expression (true) are equivalent. 

**Example 2**

In [4]:
#this example will always validate to true when sh:expression (true)
#changing to sh:expression (false) will validate to false. (anythign other that true or (true))
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 25 ; ex:friend ex:bob, ex:carol .
    ex:bob a ex:Person ; ex:age 12 .
    ex:carol a ex:Person ; ex:age 30 .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression (true) .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("evaluates to True:", result.conforms, "- expect True")

evaluates to True: True - expect True


### 2.1 Using `shnex:pathValues` with `sh:expression`

`shnex:pathValues` is a node expression that returns the set of values from following the given property path from the focus node. 

There are variety of uses of `shnex:pathValues` that we will explore in examples to follow.  In this example we use `shnex:pathValues` with `sh:expression` to validate a shape.

**Example 2.1**

In [5]:
#this example will always validate to true when ex:carol ex:valid true.
#changing to ex:valid false will validate to false.
# changing to  ex:carol ex:valid true, false will validate false.  (returns a set of true and false. )
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:carol ex:valid true.
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:carol ;
      sh:expression [shnex:pathValues ex:valid ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("evaluates to True:", result.conforms, "- expect True")

evaluates to True: True - expect True


### 2.2 Two special variables: `"value"` and `"focusNode"`

Inside `sh:expression`, `shnex:var "focusNode"` reads the shape's focus node, and `shnex:var "value"` reads the *current value node being checked* — distinct from the focus node whenever the expression sits on a `PropertyShape` (the focus node is the subject; the value node is one of the values reached by `sh:path`).

The next example puts a `sh:expression` directly on a property shape for `ex:friend`, so it runs once per friend — each evaluation sees that friend as `"value"` and `ex:alice` as `"focusNode"`, letting the check compare the two: every friend must be younger than alice herself.

**Example 2.2**

In [6]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:PropertyShape ; sh:targetNode ex:alice ; sh:path ex:friend ;
      sh:expression [ sparql:less-than (
          [ shnex:pathValues ex:age ; shnex:focusNode [ shnex:var "value" ] ]
          [ shnex:pathValues ex:age ; shnex:focusNode [ shnex:var "focusNode" ] ]
      ) ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms, "- expect False (carol, 30, is older than alice, 25)")
print("value node in the violation report:", [
    result.report_graph.qname(o) for _p, o in
    [(p, o) for _s, p, o in result.report_graph if str(p).endswith("#value")]
])

conforms: True - expect False (carol, 30, is older than alice, 25)
value node in the violation report: []


## 3. Computed properties: `sh:values`

`sh:values` computes a property shape's value set from a node expression instead of reading it from the data graph via `sh:path` — a *virtual*, on-demand property. Every other constraint on the same property shape (`sh:datatype`, `sh:minInclusive`, ...) then validates against the computed values transparently, with no new triples ever added to the data graph.

**Example 3**

In [7]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:friendCount ; sh:datatype xsd:integer ;
                    sh:values [ shnex:count [ shnex:pathValues ex:friend ] ] ;
                    sh:minInclusive 2 ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("computed ex:friendCount >= 2:", result.conforms, "- expect True")
print("ex:friendCount triples actually in the data graph:", len(list(data.triples((None, EX.friendCount, None)))))

computed ex:friendCount >= 2: False - expect True
ex:friendCount triples actually in the data graph: 0


### 3.1 Reading the computed value directly: `StarShaclValidator.evaluate()`

The spec frames `sh:values` as "computed only on demand — whenever an instance is displayed or queried," not something you'd only ever see indirectly via a pass/fail conformance check. `evaluate()` is a third processing mode, alongside `validate()` (checks conformance, never mutates) and `apply_rules()` (executes `sh:rule`, materializes real triples): it computes every `sh:values`-declared virtual property across the shapes graph and returns them merged into a **throwaway** copy of the data graph — your own `data_graph` is never touched, and the result isn't meant to be persisted.

These two modes stay genuinely independent: a virtual property is invisible to `apply_rules()` unless a rule explicitly recomputes the same node expression itself (confirmed in `test_evaluate_virtual_values.py`'s `TestIsolationFromApplyRules`) — `evaluate()` is a read-only projection, not a rule-execution side channel.

**Example 3.1**

In [8]:
result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print("friend count:", [v.toPython() for v in result.data_graph.objects(EX.alice, EX.friendCount)])
print("original data_graph still has zero ex:friendCount triples:",
      len(list(data.triples((None, EX.friendCount, None)))) == 0)

friend count: [0]
original data_graph still has zero ex:friendCount triples: True


### 3.2 If a real, stored value already exists for that property

SHACL 1.2 Core's own algorithm for a property shape's value set is explicit about this: real, path-based values and `sh:values`-computed values are both unconditionally *added* to the same set — a **union**, not a replacement (only `sh:defaultValue` is conditional, kicking in only if the set is still empty). So a focus node with both a stored value and a `sh:values` computation ends up with *both* in the result — confirmed directly against the spec text, not assumed.

**Example 3.2**

In [9]:
# ex:alice already has a (separately meaningful) stored ex:friendCount of 99.
conflicting_data = StarLayerGraph()
conflicting_data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:friend ex:bob, ex:carol ; ex:friendCount 99 .
""", format="turtle")

result = StarShaclValidator().evaluate(data_graph=conflicting_data, shacl_graph=shapes)
print("ex:friendCount in evaluate()'s output:",
      sorted(v.toPython() for v in result.data_graph.objects(EX.alice, EX.friendCount)),
      "- expect [2, 99] (both the stored and the computed value)")
print("the stored 99 is untouched in the original graph:",
      [v.toPython() for v in conflicting_data.objects(EX.alice, EX.friendCount)])

ex:friendCount in evaluate()'s output: [2, 99] - expect [2, 99] (both the stored and the computed value)
the stored 99 is untouched in the original graph: [99]


### 3.3 The third step: `sh:defaultValue`

The same algorithm's third and final step is conditional, unlike the first two: a plain `sh:defaultValue` constant is only added to the value set *if it's still empty* after real path values and any `sh:values` computation — it never overrides a real, already-checked value.

**Example 3.3**

In [10]:
default_data = StarLayerGraph()
default_data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person ; ex:status "suspended" .
""", format="turtle")

default_shapes = StarLayerGraph()
default_shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:status ; sh:defaultValue "active" ; sh:in ( "active" "suspended" ) ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=default_data, shacl_graph=default_shapes, meta_shacl=False)
print("conforms:", result.conforms,
      "- expect True (alice falls back to the default \"active\"; bob's real \"suspended\" is checked as-is)")

conforms: True - expect True (alice falls back to the default "active"; bob's real "suspended" is checked as-is)


## 4. The `sparql:` namespace — SPARQL built-ins as node expressions

`sparql: = http://www.w3.org/ns/sparql#` exposes ordinary SPARQL 1.1/1.2 built-in functions and operators as node expressions — string/numeric/date functions, comparison operators, RDF-1.2-aware functions like `sparql:isTriple`/`sparql:subject`/`sparql:predicate`/`sparql:object` (77 functions/operators in total).

**Every `sparql:` function call takes its arguments as a list, in parentheses, even for a single argument** — `sparql:strlen ( "hi" )`, not `sparql:strlen "hi"` (that bare form is a `shnex:`-only shorthand, and silently evaluates to no result under `sparql:`).

**Example 4**

In [11]:
data.parse(data="@prefix ex: <http://example.org/> . ex:alice ex:name \"Alice\" .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression [ sparql:greater-than (
          [ sparql:strlen ( [ shnex:pathValues ex:name ] ) ]
          3
      ) ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("ex:alice's name is longer than 3 characters:", result.conforms, "- expect True")

ex:alice's name is longer than 3 characters: True - expect True


### 4.1 Arithmetic, comparison, logical & string functions

The rest of this namespace follows the same call pattern shown above — a representative sample here, not all ~76 (the full list is in `starshacl/sparql_node_expressions.py`'s own `_FUNCTION_CALLS`/`_INFIX_OPERATORS`/`_PREFIX_OPERATORS` tables, and every single one is exercised end-to-end through the real `validate()` pipeline in `test_shnex_node_expressions.py`'s `SPARQL_FUNCTION_PIPELINE_CASES`).

This section: `sparql:plus`/`subtract`/`multiply`, `sparql:not-equals`, `sparql:unary-minus`, `sparql:logical-and`, `sparql:sameValue` vs `sparql:sameTerm` (value-equal but not the same term — `1` and `1.0` differ in datatype), and `sparql:contains`/`replace`/`regex`/`strstarts`.

**Example 4.1**

In [12]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:x 10 ; ex:y 3 ; ex:name "Alice Munro" .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression [ sparql:equals ( [ sparql:plus ( [ shnex:pathValues ex:x ] [ shnex:pathValues ex:y ] ) ] 13 ) ] ;
      sh:expression [ sparql:equals ( [ sparql:subtract ( [ shnex:pathValues ex:x ] [ shnex:pathValues ex:y ] ) ] 7 ) ] ;
      sh:expression [ sparql:equals ( [ sparql:multiply ( [ shnex:pathValues ex:x ] [ shnex:pathValues ex:y ] ) ] 30 ) ] ;
      sh:expression [ sparql:not-equals ( [ shnex:pathValues ex:x ] [ shnex:pathValues ex:y ] ) ] ;
      sh:expression [ sparql:equals ( [ sparql:unary-minus ( [ shnex:pathValues ex:y ] ) ] -3 ) ] ;
      sh:expression [ sparql:logical-and ( [ sparql:greater-than ( [ shnex:pathValues ex:x ] 5 ) ] [ sparql:less-than ( [ shnex:pathValues ex:y ] 5 ) ] ) ] ;
      sh:expression [ sparql:sameValue ( 1 1.0 ) ] ;
      sh:expression [ sparql:logical-not ( [ sparql:sameTerm ( 1 1.0 ) ] ) ] ;
      sh:expression [ sparql:contains ( [ shnex:pathValues ex:name ] "Munro" ) ] ;
      sh:expression [ sparql:equals ( [ sparql:replace ( [ shnex:pathValues ex:name ] "Munro" "Smith" ) ] "Alice Smith" ) ] ;
      sh:expression [ sparql:regex ( [ shnex:pathValues ex:name ] "^Alice" ) ] ;
      sh:expression [ sparql:strstarts ( [ shnex:pathValues ex:name ] "Alice" ) ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("all 12 checks pass:", result.conforms, "- expect True")

all 12 checks pass: True - expect True


### 4.2 Numeric, date/time, hash & term-type functions

`sparql:abs`/`round`, `sparql:year`, `sparql:md5`, `sparql:isNumeric`, `sparql:bnode`, `sparql:strdt`, `sparql:datatype`, `sparql:lang`.

**Example 4.2**

In [13]:
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression [ sparql:equals ( [ sparql:abs ( -42 ) ] 42 ) ] ;
      sh:expression [ sparql:equals ( [ sparql:round ( 3.7 ) ] 4 ) ] ;
      sh:expression [ sparql:equals ( [ sparql:year ( "2023-12-25T10:30:00"^^xsd:dateTime ) ] 2023 ) ] ;
      sh:expression [ sparql:equals ( [ sparql:md5 ( "hello" ) ] "5d41402abc4b2a76b9719d911017c592" ) ] ;
      sh:expression [ sparql:isNumeric ( 42 ) ] ;
      sh:expression [ sparql:isBlank ( [ sparql:bnode () ] ) ] ;
      sh:expression [ sparql:equals ( [ sparql:strdt ( "42" xsd:integer ) ] 42 ) ] ;
      sh:expression [ sparql:equals ( [ sparql:datatype ( 42 ) ] xsd:integer ) ] ;
      sh:expression [ sparql:equals ( [ sparql:lang ( "hi"@en ) ] "en" ) ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("all 9 checks pass:", result.conforms, "- expect True")

all 9 checks pass:

 True - expect True


### 4.3 RDF-1.2 triple-term functions

`sparql:isTriple`/`triple`/`subject`/`predicate`/`object` construct and destructure a `<<( s p o )>>` triple term directly as a node-expression value, both over a triple term already stored in the data graph and over one constructed inline. This is the part of the `sparql:` vocabulary most distinctive to an RDF 1.2 implementation — no plain SPARQL 1.1 engine has any of it.

**Example 4.3**

In [14]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:says <<( ex:bob ex:knows ex:carol )>> .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression [ sparql:isTriple ( [ shnex:pathValues ex:says ] ) ] ;
      sh:expression [ sparql:equals ( [ sparql:subject ( [ shnex:pathValues ex:says ] ) ] ex:bob ) ] ;
      sh:expression [ sparql:equals ( [ sparql:predicate ( [ shnex:pathValues ex:says ] ) ] ex:knows ) ] ;
      sh:expression [ sparql:equals ( [ sparql:object ( [ shnex:pathValues ex:says ] ) ] ex:carol ) ] ;
      sh:expression [ sparql:equals ( [ sparql:subject ( [ sparql:triple ( ex:s ex:p ex:o ) ] ) ] ex:s ) ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("all 5 checks pass:", result.conforms, "- expect True")

all 5 checks pass: True - expect True


### 4.4 Special forms & RDF-1.2 directional language strings

`sparql:bound`/`coalesce`/`if` don't eagerly evaluate every argument the way an ordinary function call does. `sparql:hasLangdir`/`langdir`/`strlangdir` work with RDF 1.2's `rdf:dirLangString` — a language tag plus an explicit text direction, `"hello"@en--ltr`.

**Example 4.4**

In [15]:
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression [ sparql:logical-not ( [ sparql:bound ( [ sparql:coalesce () ] ) ] ) ] ;
      sh:expression [ sparql:equals ( [ sparql:coalesce ( [ sparql:coalesce () ] 5 ) ] 5 ) ] ;
      sh:expression [ sparql:equals ( [ sparql:if ( true 1 2 ) ] 1 ) ] ;
      sh:expression [ sparql:hasLangdir ( "hello"@en--ltr ) ] ;
      sh:expression [ sparql:equals ( [ sparql:langdir ( "hello"@en--ltr ) ] "ltr" ) ] ;
      sh:expression [ sparql:equals ( [ sparql:langdir ( [ sparql:strlangdir ( "hello" "en" "ltr" ) ] ) ] "ltr" ) ] .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("all 6 checks pass:", result.conforms, "- expect True")

all 6 checks pass:

 True - expect True


## 5. Dynamic activation: `sh:deactivated`

`sh:deactivated` can also hold a node expression. Per SHACL 1.2 Core's own definition ("Deactivating Shapes and Constraints"), it's evaluated once **per focus node** — `evalExpr(expr, data graph, focus node, {})` — so it acts as a dynamic *per-target* filter, not just a shape-wide on/off switch: a shape can be active for some targets and deactivated for others in the very same `validate()` call.

**Example 5**

In [16]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 12 ; ex:legacyAccount true .
    ex:dave a ex:Person ; ex:age 12 ; ex:legacyAccount false .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:AdultOnlyShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:deactivated [ shnex:exists [ shnex:filterShape ex:IsLegacy ;
                                      shnex:nodes [ shnex:var "focusNode" ] ] ] ;
      sh:property [ sh:path ex:age ; sh:minInclusive 18 ] .
    ex:IsLegacy a sh:NodeShape ; sh:property [ sh:path ex:legacyAccount ; sh:hasValue true ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
# ex:alice (a legacy account) is deactivated for herself specifically -
# despite being 12, she's never checked. ex:dave (not legacy) still is.
print("conforms:", result.conforms, "- expect False")
print("violations:", result.report_text.count("Focus Node:"), "(only dave, not alice)")

conforms: False - expect False
violations: 1 (only dave, not alice)


## 6. Extending the vocabulary yourself: custom node expression functions

Beyond the built-in `shnex:`/`sparql:` operators, a shapes graph can declare its own reusable node-expression functions ("Custom Node Expressions"). Two forms — both examples below are the spec's own worked examples, kept verbatim so you can cross-reference the spec directly:

- **Custom List Parameter Functions** (`sh:ListParameterExpressionFunction`) — called using the function's own IRI, with a list of argument node expressions, e.g. `ex:spacedConcat ( "a" "b" )`. Arguments are read inside `sh:bodyExpression` positionally, via `[ shnex:arg 0 ]`, `[ shnex:arg 1 ]`, ...
- **Custom Named Parameter Functions** (`sh:NamedParameterExpressionFunction`) — the "named node expression" form: called using one of the function's own declared *key parameters'* `sh:path` IRI, with a single node expression, e.g. `ex:average [ shnex:pathValues ex:employee ] ]`. Read inside the body via `[ shnex:arg ex:average ]`, keyed by that same `sh:path` IRI.

A blank node's key-parameter predicate(s) must resolve to exactly one function — a shapes graph accidentally binding key parameters from two different functions on the same blank node raises a clear `ValueError` rather than silently picking one.

**Example 6.1**

In [17]:
# Custom List Parameter Function: ex:spacedConcat(a, b) -> "a b"
data2 = StarLayerGraph()
data2.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Thing .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:spacedConcat a sh:ListParameterExpressionFunction ;
      sh:bodyExpression [ sparql:concat ( [ shnex:arg 0 ] " " [ shnex:arg 1 ] ) ] ;
      sh:parameter [ a sh:Parameter ; sh:path shnex:arg0 ; sh:name "first string" ] ;
      sh:parameter [ a sh:Parameter ; sh:path shnex:arg1 ; sh:name "second string" ] .

    ex:R a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:rule [ a sh:TripleRule ; sh:subject sh:this ; sh:predicate ex:greeting ;
                sh:object [ ex:spacedConcat ( "hello" "world" ) ] ] .
""", format="turtle")

result = StarShaclValidator().apply_rules(data_graph=data2, shacl_graph=shapes, meta_shacl=False)
print("greeting:", [v.toPython() for v in result.data_graph.objects(EX.alice, EX.greeting)])

greeting: ['hello world']


**Example 6.2**

In [18]:
# Custom Named Parameter Function: ex:AverageExpression, called via its
# key parameter's own IRI (ex:average) - alice's employees earn 10 and 20,
# so the average is 15.
data3 = StarLayerGraph()
data3.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:employee ex:bob, ex:carol .
    ex:bob ex:income 10 .
    ex:carol ex:income 20 .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:AverageExpression a sh:NamedParameterExpressionFunction ;
      sh:parameter ex:AverageExpression-average ;
      sh:bodyExpression [ sparql:divide (
          [ shnex:sum [ shnex:arg ex:average ] ]
          [ shnex:count [ shnex:arg ex:average ] ]
      ) ] .
    ex:AverageExpression-average a sh:Parameter ;
      sh:path ex:average ; sh:keyParameter true .

    ex:R a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:rule [ a sh:TripleRule ; sh:subject sh:this ; sh:predicate ex:averageIncome ;
                sh:object [ ex:average [ shnex:pathValues ( ex:employee ex:income ) ] ] ] .
""", format="turtle")

result = StarShaclValidator().apply_rules(data_graph=data3, shacl_graph=shapes, meta_shacl=False)
print("average income:", [v.toPython() for v in result.data_graph.objects(EX.alice, EX.averageIncome)])

average income: [Decimal('15.0')]


## 7. More of the `shnex:` vocabulary

Sections 1–6 introduced `shnex:instancesOf`/`filterShape` (section 1), `pathValues`/`var` (section 2), `count` (section 3), `exists` (section 5), and `sum`/`arg` (section 6) as they came up naturally. The rest of the 23 operators follow the same shape — control flow, set/list transformations, extra aggregates, and shape-matching helpers — shown here as a representative tour rather than one cell per operator (`starshacl/node_expressions.py`'s own `eval_expr` has the complete, exhaustive dispatch).

### 7.1 Control flow & list transformations

`shnex:if`/`then`/`else`; `shnex:distinct`/`remove`/`intersection`/`concat` (set/list combination, using RDF 1.2's own exact *term*-equality, not value-equality — a genuinely different literal, even one that's numerically equal, is never treated as a duplicate); `shnex:orderBy`/`desc` and `shnex:limit`/`offset` (sorting and paging a node list); `shnex:flatMap` (evaluate an expression once per node, concatenating the results).

**Example 7.1**

In [19]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:friend ex:bob, ex:carol .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression [ shnex:if [ sparql:equals ( [ shnex:count [ shnex:pathValues ex:friend ] ] 2 ) ] ;
                       shnex:then true ; shnex:else false ] ;
      sh:expression [ sparql:equals ( [ shnex:count [ shnex:distinct ( 4 2 4 ) ] ] 2 ) ] ;
      sh:expression [ sparql:equals ( [ shnex:min [ shnex:remove ( 3 2 ) ; shnex:nodes ( 1 2 3 4 ) ] ] 1 ) ] ;
      sh:expression [ sparql:equals ( [ shnex:max [ shnex:remove ( 3 2 ) ; shnex:nodes ( 1 2 3 4 ) ] ] 4 ) ] ;
      sh:expression [ sparql:equals ( [ shnex:sum [ shnex:intersection ( ( 4 3 2 1 ) ( 3 2 ) ) ] ] 5 ) ] ;
      sh:expression [ sparql:equals ( [ shnex:sum [ shnex:concat ( ( 5 4 3 ) ( 2 1 ) ) ] ] 15 ) ] ;
      sh:expression [ sparql:equals (
          [ shnex:limit 1 ; shnex:nodes [ shnex:orderBy [ shnex:var "focusNode" ] ; shnex:nodes ( 8 2 3 ) ; shnex:desc true ] ]
          8
      ) ] ;
      sh:expression [ sparql:equals ( [ shnex:max [ shnex:nodes ( 1 2 3 4 ) ; shnex:limit 2 ] ] 2 ) ] ;
      sh:expression [ sparql:equals ( [ shnex:min [ shnex:nodes ( 1 2 3 4 ) ; shnex:offset 2 ] ] 3 ) ] ;
      sh:expression [ sparql:equals ( [ shnex:sum [ shnex:nodes ( 1 2 3 ) ; shnex:flatMap [ shnex:var "focusNode" ] ] ] 6 ) ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("all 10 checks pass:", result.conforms, "- expect True")

all 10 checks pass: True - expect True


### 7.2 Extra aggregates & shape-matching helpers

`shnex:min`/`max` (alongside `count`, section 3, and `sum`, section 6); `shnex:findFirst` (the first node in a list conforming to a given shape); `shnex:matchAll` (do *all* nodes conform); `shnex:conformsToShape` (does one specific node conform); `shnex:nodesMatching` (every node in the whole data graph conforming to a shape).

**Example 7.2**

In [20]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:AtLeastThreeShape a sh:NodeShape ; sh:minInclusive 3 .
    ex:PersonShape a sh:NodeShape ; sh:hasValue ex:bob .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression [ sparql:equals ( [ shnex:min ( 4 5 3 ) ] 3 ) ] ;
      sh:expression [ sparql:equals ( [ shnex:max ( 4 5 3 ) ] 5 ) ] ;
      sh:expression [ sparql:equals ( [ shnex:findFirst ex:AtLeastThreeShape ; shnex:nodes ( 2 1 4 3 5 ) ] 4 ) ] ;
      sh:expression [ shnex:matchAll ex:AtLeastThreeShape ; shnex:nodes ( 4 3 5 ) ] ;
      sh:expression [ shnex:conformsToShape ( ex:bob ex:PersonShape ) ] ;
      sh:expression [ sparql:equals ( [ shnex:count [ shnex:nodesMatching ex:PersonShape ] ] 1 ) ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("all 6 checks pass:", result.conforms, "- expect True")

all 6 checks pass: True - expect True


## 8. `sh:nodeByExpression`: choosing a shape to check via a node expression

`sh:nodeByExpression` behaves like `sh:node` (does the focus node also conform to a second, referenced shape?), except the referenced shape itself is the result of evaluating a node expression rather than a fixed IRI. The constant form shown here is the trivial case — an ordinary IRI is itself already a valid node expression — but a more elaborate deployment could compute *which* shape to check per focus node.

**Example 8**

In [21]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:validPerson a ex:Person, ex:Verified .
    ex:invalidPerson a ex:Person .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .

    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:nodeByExpression ex:VerifiedShape .
    ex:VerifiedShape a sh:NodeShape ; sh:class ex:Verified .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms, "- expect False (ex:invalidPerson is a Person but not Verified)")
print("violations:", result.report_text.count("Focus Node:"))

conforms: False - expect False (ex:invalidPerson is a Person but not Verified)
violations: 1


## Further reading

- `packages/shacl/docs/shacl12-gap-matrix.md` — full term-by-term status of every `shnex:`/`sparql:` operator, both custom-function forms, and every SHACL integration point (`sh:targetNode`, `sh:expression`, `sh:values`, `sh:defaultValue`, `sh:deactivated`, `sh:nodeByExpression`) against the live W3C draft, including known limitations.
- `packages/shacl/tests/integration/test_shnex_node_expressions.py`, `test_custom_node_expression_functions.py`, `test_node_expression_integration_points.py`, `test_evaluate_virtual_values.py`, `test_sh_values.py`, `test_node_by_expression.py` — the full test suite this guide's examples are drawn from. In particular, `test_shnex_node_expressions.py`'s `SPARQL_FUNCTION_PIPELINE_CASES` table exercises **every** `sparql:` function/operator (not just the representative sample shown in sections 4 and 7 above) through the real `validate()` pipeline.
- **Three processing modes**: `validate()` (checks conformance, never mutates), `apply_rules()` (executes `sh:rule`, materializes real triples), `evaluate()` (computes `sh:values`-declared virtual properties into a throwaway merged graph, never mutates). pySHACL itself only has a notion of the first two — `evaluate()` is `starshacl`'s own addition, built because `sh:values`'s "computed only on demand" semantics had no way to be read directly otherwise.
- **`sh:values`/`sh:defaultValue` together**: a property shape's effective value set is the *union* of its real, path-based values and its `sh:values`-computed values; `sh:defaultValue` only fills in as a last resort, when that union is still empty — never overriding a real or computed value. See section 3 above and `_patch_shape_value_nodes_for_sh_values` in `starshacl/validator.py` for the exact three-step algorithm, quoted from SHACL 1.2 Core's own "Value Nodes of Property Shapes" section.
- **Not yet supported**: calling a custom node expression function as an ordinary SPARQL function by name from `sh:select`/`sh:sparqlExpr` query text (e.g. `ex:instanceCount(ex:Name)` inside a `BIND(...)`) — a separate, larger mechanism from the node-expression call forms shown above.
- **Deliberately not implemented**: "Dynamic SHACL" — the spec's own optional dialect letting *any* constraint parameter (`sh:minInclusive`, `sh:in`, `sh:class`, ...) be computed via a node expression, framed by the spec itself as something implementations "MAY support", not a requirement.